# K5 open-set diagnostic: allow unsupported pixels to remain unclassified

This notebook tests whether the K5 classifier can improve its **selective predictive performance** by abstaining on pixels that are poor fits to all five empirical class distributions.

The current random forest is closed-set: every valid pixel must receive one of the five K5 labels. Here we add class-conditional support geometry in annual-z IA45 space using Ledoit-Wolf shrinkage covariance and empirical cross-fitted Mahalanobis-distance thresholds.

For class \(c\),

\[
D_c^2(x)=(x-\mu_c)^T\Sigma_c^{-1}(x-\mu_c)
\]

and define

\[
R_c(x)=D_c^2(x)/T_c,
\]

where \(T_c\) is an empirical class-specific support threshold estimated from cross-fitted within-class distances.

A pixel is supported by class \(c\) when \(R_c \le 1\). A pixel is eligible for rejection when

\[
\min_c R_c(x)>1,
\]

meaning that it falls outside empirical support for **all five** K5 states.

The notebook uses the current `augmented_all_2026` diagnostic support, evaluates rejection on the same grouped held-out baseline observations, sweeps several support quantiles, and maps:

- number of K5 supports containing each pixel;
- minimum support ratio;
- classifier/support disagreement;
- K5 classification with `6 = UNCLASSIFIED`.

The reject option is evaluated using a risk–coverage framework. Higher accuracy after rejection is only useful if achieved at reasonable retained coverage.

In [ ]:
from pathlib import Path
import time
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import rasterio
import rasterio.mask

from rasterio.transform import array_bounds
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

from sklearn.base import clone
from sklearn.covariance import LedoitWolf
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold, KFold
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

pd.set_option("display.max_columns", 250)
pd.set_option("display.width", 320)

# -----------------------------------------------------------------------------
# INPUTS
# -----------------------------------------------------------------------------

BASELINE_DATASET = Path(
    r"C:\NCA_DATA\Analysis\PrelimML_AnnualZScore_IA45_K5"
    r"\FieldVeg_K5_AnnualZScore_IA45_modeling_dataset.csv"
)

RESOLVED_DIAGNOSTIC_ROOT = Path(
    r"C:\NCA_DATA\Analysis\Diagnostics"
    r"\K5_2026Veg_as2025Optics_RESOLVED_v1"
)

PSEUDO_2026_DATASET = (
    RESOLVED_DIAGNOSTIC_ROOT
    / "Tables"
    / "DIAGNOSTIC_2026veg_sampled_at_2025_IA45_z.csv"
)

EXISTING_CLASS_MAP = (
    RESOLVED_DIAGNOSTIC_ROOT
    / "Maps"
    / "DIAGNOSTIC_augmented_all_2026_2025_class.tif"
)

EXISTING_UNCERTAINTY_MAP = (
    RESOLVED_DIAGNOSTIC_ROOT
    / "Maps"
    / "DIAGNOSTIC_augmented_all_2026_2025_uncertainty.tif"
)

HARMONIC_ROOT = Path(
    r"A:\NCA_DATA\S2_Harmonics"
)

OPTICAL_YEAR = 2025

ZSCORE_PARAMETER_CSV = Path(
    r"C:\NCA_DATA\Analysis\PrelimML_AnnualZScore_IA45"
    r"\annual_IA45_zscore_parameters_S500.csv"
)

TRUE_NCA_BOUNDARY = Path(
    r"C:\NCA_DATA\templates\SRBOP_NCA_extent_26911.shp"
)

OUTPUT_DIR = Path(
    r"C:\NCA_DATA\Analysis\Diagnostics"
    r"\K5_OpenSet_RejectOption_LedoitWolf_v1"
)

TABLE_DIR = OUTPUT_DIR / "Tables"
MAP_DIR = OUTPUT_DIR / "Maps"
FIG_DIR = OUTPUT_DIR / "Figures"

ALLOW_OVERWRITE = False

for d in [OUTPUT_DIR, TABLE_DIR, MAP_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

existing = [p for p in OUTPUT_DIR.rglob("*") if p.is_file()]
if existing and not ALLOW_OVERWRITE:
    raise FileExistsError(
        "Open-set diagnostic directory already contains files.\n"
        "Nothing will be overwritten.\n\n"
        f"{OUTPUT_DIR}"
    )

K5_LABELS = {
    1: "PBG",
    2: "ARTR",
    3: "Non-ARTR shrub",
    4: "Exotic forb",
    5: "EAG",
}

K5_CODES = [1, 2, 3, 4, 5]
UNCLASSIFIED_CODE = 6

RANDOM_STATE = 42
N_SPLITS = 5

RF_TEMPLATE = RandomForestClassifier(
    n_estimators=800,
    max_features="sqrt",
    min_samples_leaf=2,
    class_weight="balanced_subsample",
    bootstrap=True,
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

SUPPORT_QUANTILES = [0.90, 0.95, 0.975, 0.99]
DEFAULT_SUPPORT_QUANTILE = 0.95
SUPPORT_INNER_FOLDS = 5
OOF_SPATIAL_GUARD_M = 100.0

COMPRESS = "DEFLATE"
DISPLAY_MAPS = True

print("K5 OPEN-SET / REJECT-OPTION DIAGNOSTIC")
print("Output:", OUTPUT_DIR)

In [ ]:
# =============================================================================
# HELPERS
# =============================================================================

def annual_k2_path(year):
    year = int(year)
    return (
        HARMONIC_ROOT
        / str(year)
        / f"{year}_K2_InterceptAmpPhase_nobs_RMSE.tif"
    )


def normalize_feature_name(x):
    return str(x).strip().lower()


def get_band_names(src):
    return [
        str(desc).strip() if desc is not None and str(desc).strip() else f"band_{i}"
        for i, desc in enumerate(src.descriptions, start=1)
    ]


def ia_band_indexes(src, feature_names):
    names = get_band_names(src)
    lookup = {
        normalize_feature_name(name): i
        for i, name in enumerate(names, start=1)
    }

    indexes = []
    missing = []

    for feature in feature_names:
        key = normalize_feature_name(feature)

        if key not in lookup:
            missing.append(feature)
        else:
            indexes.append(lookup[key])

    if missing:
        raise KeyError(
            "Missing IA45 bands in annual raster:\n"
            + "\n".join(missing)
        )

    if len(indexes) != 45:
        raise RuntimeError(
            f"Expected 45 IA45 raster bands; found {len(indexes)}."
        )

    return indexes


def mahalanobis_squared(X, mean, precision):
    diff = X - mean

    return np.einsum(
        "ij,jk,ik->i",
        diff,
        precision,
        diff,
        optimize=True,
    )


def fit_class_support_model(
    X_class,
    support_quantile,
    random_state=RANDOM_STATE,
    inner_folds=SUPPORT_INNER_FOLDS,
):
    X_class = np.asarray(
        X_class,
        dtype=np.float64,
    )

    n = len(X_class)

    if n < 6:
        raise ValueError(
            f"Too few observations ({n}) to estimate class support."
        )

    n_splits = min(inner_folds, n)

    kf = KFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state,
    )

    crossfit_d2 = np.full(
        n,
        np.nan,
        dtype=np.float64,
    )

    for train_idx, test_idx in kf.split(X_class):
        lw = LedoitWolf().fit(
            X_class[train_idx]
        )

        crossfit_d2[test_idx] = mahalanobis_squared(
            X_class[test_idx],
            lw.location_,
            lw.precision_,
        )

    if not np.isfinite(crossfit_d2).all():
        raise RuntimeError(
            "Non-finite cross-fitted Mahalanobis distances."
        )

    threshold = float(
        np.quantile(
            crossfit_d2,
            support_quantile,
        )
    )

    full_model = LedoitWolf().fit(X_class)

    return {
        "mean": full_model.location_.astype(np.float64),
        "precision": full_model.precision_.astype(np.float64),
        "threshold": threshold,
        "crossfit_d2": crossfit_d2,
    }


def fit_support_models(X, y, support_quantile):
    models = {}

    for code_i in K5_CODES:
        mask = y == code_i

        models[code_i] = fit_class_support_model(
            X[mask],
            support_quantile=support_quantile,
            random_state=RANDOM_STATE + code_i,
        )

    return models


def support_diagnostics(X, rf_pred, support_models):
    n = X.shape[0]

    d2_matrix = np.empty(
        (n, len(K5_CODES)),
        dtype=np.float64,
    )

    threshold_vector = np.empty(
        len(K5_CODES),
        dtype=np.float64,
    )

    for j, code_i in enumerate(K5_CODES):
        model = support_models[code_i]

        d2_matrix[:, j] = mahalanobis_squared(
            X,
            model["mean"],
            model["precision"],
        )

        threshold_vector[j] = model["threshold"]

    ratio_matrix = (
        d2_matrix
        / threshold_vector[None, :]
    )

    support_matrix = ratio_matrix <= 1.0

    support_count = (
        support_matrix
        .sum(axis=1)
        .astype(np.uint8)
    )

    min_idx = np.argmin(
        ratio_matrix,
        axis=1,
    )

    min_support_ratio = ratio_matrix[
        np.arange(n),
        min_idx,
    ]

    nearest_support_class = np.asarray(
        K5_CODES,
        dtype=np.uint8,
    )[min_idx]

    class_to_col = {
        code_i: j
        for j, code_i in enumerate(K5_CODES)
    }

    pred_cols = np.array(
        [
            class_to_col[int(code_i)]
            for code_i in rf_pred
        ],
        dtype=np.int16,
    )

    predicted_supported = support_matrix[
        np.arange(n),
        pred_cols,
    ]

    status = np.full(
        n,
        1,
        dtype=np.uint8,
    )

    status[
        (support_count > 0)
        & (~predicted_supported)
    ] = 2

    status[
        support_count == 0
    ] = 3

    return {
        "ratio_matrix": ratio_matrix,
        "support_matrix": support_matrix,
        "support_count": support_count,
        "min_support_ratio": min_support_ratio,
        "nearest_support_class": nearest_support_class,
        "predicted_supported": predicted_supported,
        "status": status,
    }


def pooled_scores(y_true, y_pred):
    return {
        "accuracy": accuracy_score(
            y_true,
            y_pred,
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_true,
            y_pred,
        ),
        "macro_f1": f1_score(
            y_true,
            y_pred,
            labels=K5_CODES,
            average="macro",
            zero_division=0,
        ),
    }


def selective_scores(y_true, y_pred, accepted):
    accepted = np.asarray(
        accepted,
        dtype=bool,
    )

    rejected = ~accepted

    n = len(y_true)
    n_accept = int(accepted.sum())
    n_reject = int(rejected.sum())

    correct = y_pred == y_true
    total_errors = int((~correct).sum())

    rejected_errors = int(
        (
            rejected
            & (~correct)
        ).sum()
    )

    if n_accept > 0:
        selective_accuracy = accuracy_score(
            y_true[accepted],
            y_pred[accepted],
        )

        selective_macro_f1 = f1_score(
            y_true[accepted],
            y_pred[accepted],
            labels=K5_CODES,
            average="macro",
            zero_division=0,
        )
    else:
        selective_accuracy = np.nan
        selective_macro_f1 = np.nan

    if n_reject > 0:
        rejected_error_rate = float(
            np.mean(
                y_pred[rejected]
                != y_true[rejected]
            )
        )
    else:
        rejected_error_rate = np.nan

    return {
        "coverage": n_accept / n,
        "rejection_rate": n_reject / n,
        "selective_accuracy": selective_accuracy,
        "selective_macro_f1": selective_macro_f1,
        "rejected_error_rate": rejected_error_rate,
        "fraction_all_errors_captured_by_rejection": (
            rejected_errors / total_errors
            if total_errors > 0
            else np.nan
        ),
        "n_accepted": n_accept,
        "n_rejected": n_reject,
    }


def grids_match(src_a, src_b):
    return (
        src_a.crs == src_b.crs
        and src_a.width == src_b.width
        and src_a.height == src_b.height
        and src_a.transform.almost_equals(src_b.transform)
    )


def write_csv_safe(df, path):
    path = Path(path)

    if path.exists() and not ALLOW_OVERWRITE:
        raise FileExistsError(path)

    df.to_csv(
        path,
        index=False,
    )

    print("Wrote:", path)

## Load the augmented-all training support

The full class-support geometry is learned from the original contemporaneous baseline observations plus all valid pseudo-2026 observations from the resolved support diagnostic.

In [ ]:
# =============================================================================
# LOAD DATA + 2025 FEATURE SCHEMA
# =============================================================================

required_paths = [
    BASELINE_DATASET,
    PSEUDO_2026_DATASET,
    ZSCORE_PARAMETER_CSV,
    annual_k2_path(OPTICAL_YEAR),
    EXISTING_CLASS_MAP,
    EXISTING_UNCERTAINTY_MAP,
    TRUE_NCA_BOUNDARY,
]

for path in required_paths:
    if not path.exists():
        raise FileNotFoundError(path)


baseline = pd.read_csv(
    BASELINE_DATASET,
    low_memory=False,
).drop(
    columns=["Unnamed: 0"],
    errors="ignore",
)

pseudo = pd.read_csv(
    PSEUDO_2026_DATASET,
    low_memory=False,
).drop(
    columns=["Unnamed: 0"],
    errors="ignore",
)


baseline["parent_cluster_k5"] = pd.to_numeric(
    baseline["parent_cluster_k5"],
    errors="raise",
).astype(int)

pseudo["parent_cluster_k5"] = pd.to_numeric(
    pseudo["parent_cluster_k5"],
    errors="raise",
).astype(int)


Z_COLS = [
    c
    for c in baseline.columns
    if str(c).startswith("z__")
]

if len(Z_COLS) != 45:
    raise ValueError(
        f"Expected 45 baseline z-score predictors; found {len(Z_COLS)}."
    )

missing_z = [
    c
    for c in Z_COLS
    if c not in pseudo.columns
]

if missing_z:
    raise KeyError(
        f"Pseudo-2026 dataset missing {len(missing_z)} z-score columns."
    )


params = pd.read_csv(
    ZSCORE_PARAMETER_CSV
)

params["Year"] = pd.to_numeric(
    params["Year"],
    errors="raise",
).astype(int)

p2025 = (
    params.loc[
        params["Year"].eq(OPTICAL_YEAR)
    ]
    .copy()
)

SAMPLE_IA_COLS = (
    p2025["feature"]
    .astype(str)
    .tolist()
)

expected_z_cols = [
    f"z__{feature}"
    for feature in SAMPLE_IA_COLS
]

if set(expected_z_cols) != set(Z_COLS):
    raise ValueError(
        "Baseline z-score columns do not match the saved 2025 IA45 schema."
    )

Z_COLS = expected_z_cols

p2025 = (
    p2025
    .set_index("feature")
    .loc[SAMPLE_IA_COLS]
)

MU_2025 = p2025[
    "weighted_mean"
].to_numpy(
    dtype=np.float64
)

SD_2025 = p2025[
    "weighted_sd"
].to_numpy(
    dtype=np.float64
)


X_base = baseline[
    Z_COLS
].to_numpy(
    dtype=np.float64
)

y_base = baseline[
    "parent_cluster_k5"
].to_numpy(
    dtype=int
)

groups_base = baseline[
    "PlotID"
].astype(str).to_numpy()

xy_base = baseline[
    ["Easting", "Northing"]
].to_numpy(
    dtype=np.float64
)


X_pseudo = pseudo[
    Z_COLS
].to_numpy(
    dtype=np.float64
)

y_pseudo = pseudo[
    "parent_cluster_k5"
].to_numpy(
    dtype=int
)

xy_pseudo = pseudo[
    ["Easting", "Northing"]
].to_numpy(
    dtype=np.float64
)


support_counts = pd.DataFrame(
    {
        "class_code": K5_CODES,
        "class_label": [
            K5_LABELS[c]
            for c in K5_CODES
        ],
        "baseline_n": [
            int(np.sum(y_base == c))
            for c in K5_CODES
        ],
        "pseudo_2026_n": [
            int(np.sum(y_pseudo == c))
            for c in K5_CODES
        ],
    }
)

support_counts["total_n"] = (
    support_counts["baseline_n"]
    + support_counts["pseudo_2026_n"]
)

display(support_counts)

print(
    "Total augmented-all support:",
    len(y_base) + len(y_pseudo)
)

## Grouped OOF rejection sweep

For each outer fold, the random forest and all class-support models are learned from training observations only. Pseudo-2026 points within 100 m of a held-out baseline observation are excluded from that fold.

The main reject rule is conservative: an observation is rejected **only if no K5 class supports it**.

In [ ]:
# =============================================================================
# GROUPED OOF REJECTION SWEEP
# =============================================================================

cv = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

splits = list(
    cv.split(
        X_base,
        y_base,
        groups=groups_base,
    )
)

oof_rf_pred = np.full(
    len(y_base),
    -1,
    dtype=np.int16,
)

oof_support = {
    q: {
        "support_count": np.full(
            len(y_base),
            255,
            dtype=np.uint8,
        ),
        "min_ratio": np.full(
            len(y_base),
            np.nan,
            dtype=np.float64,
        ),
        "predicted_supported": np.zeros(
            len(y_base),
            dtype=bool,
        ),
        "status": np.full(
            len(y_base),
            255,
            dtype=np.uint8,
        ),
    }
    for q in SUPPORT_QUANTILES
}


for fold, (train_idx, test_idx) in enumerate(
    splits,
    start=1,
):

    test_xy = xy_base[test_idx]

    nn_test = NearestNeighbors(
        n_neighbors=1
    )

    nn_test.fit(
        test_xy
    )

    d_to_test, _ = nn_test.kneighbors(
        xy_pseudo
    )

    keep_pseudo = (
        d_to_test[:, 0]
        > OOF_SPATIAL_GUARD_M
    )

    X_train = np.vstack(
        [
            X_base[train_idx],
            X_pseudo[keep_pseudo],
        ]
    )

    y_train = np.concatenate(
        [
            y_base[train_idx],
            y_pseudo[keep_pseudo],
        ]
    )

    rf = clone(
        RF_TEMPLATE
    )

    rf.fit(
        X_train,
        y_train,
    )

    pred = rf.predict(
        X_base[test_idx]
    ).astype(
        np.int16
    )

    oof_rf_pred[
        test_idx
    ] = pred

    for q in SUPPORT_QUANTILES:

        support_models = fit_support_models(
            X_train,
            y_train,
            support_quantile=q,
        )

        diagnostic = support_diagnostics(
            X_base[test_idx],
            pred,
            support_models,
        )

        oof_support[q][
            "support_count"
        ][test_idx] = diagnostic[
            "support_count"
        ]

        oof_support[q][
            "min_ratio"
        ][test_idx] = diagnostic[
            "min_support_ratio"
        ]

        oof_support[q][
            "predicted_supported"
        ][test_idx] = diagnostic[
            "predicted_supported"
        ]

        oof_support[q][
            "status"
        ][test_idx] = diagnostic[
            "status"
        ]


if np.any(oof_rf_pred < 0):
    raise RuntimeError(
        "Incomplete OOF RF predictions."
    )


baseline_oof_metrics = pooled_scores(
    y_base,
    oof_rf_pred,
)

print(
    "Original augmented-all RF OOF:"
)

print(
    baseline_oof_metrics
)

In [ ]:
# =============================================================================
# SELECTIVE PERFORMANCE / RISK-COVERAGE
# =============================================================================

selective_rows = []
class_rejection_rows = []


for q in SUPPORT_QUANTILES:

    diagnostic = oof_support[q]

    accepted = (
        diagnostic["support_count"]
        > 0
    )

    scores = selective_scores(
        y_base,
        oof_rf_pred,
        accepted,
    )

    selective_rows.append(
        {
            "support_quantile": q,
            **baseline_oof_metrics,
            **scores,
        }
    )

    for code_i in K5_CODES:

        class_mask = (
            y_base
            == code_i
        )

        class_rejection_rows.append(
            {
                "support_quantile": q,
                "class_code": code_i,
                "class_label": K5_LABELS[code_i],
                "n_true_class": int(
                    class_mask.sum()
                ),
                "rejection_rate": float(
                    np.mean(
                        ~accepted[class_mask]
                    )
                ),
                "mean_min_support_ratio": float(
                    np.mean(
                        diagnostic[
                            "min_ratio"
                        ][
                            class_mask
                        ]
                    )
                ),
            }
        )


selective_df = pd.DataFrame(
    selective_rows
)

class_rejection_df = pd.DataFrame(
    class_rejection_rows
)


write_csv_safe(
    selective_df,
    TABLE_DIR
    / "DIAGNOSTIC_OOF_selective_performance_by_support_quantile.csv",
)

write_csv_safe(
    class_rejection_df,
    TABLE_DIR
    / "DIAGNOSTIC_OOF_class_rejection_by_support_quantile.csv",
)


display(
    selective_df.round(4)
)

print(
    "\nClass-specific rejection rate:"
)

display(
    class_rejection_df.pivot(
        index="class_label",
        columns="support_quantile",
        values="rejection_rate",
    ).round(3)
)


risk = (
    selective_df[
        [
            "support_quantile",
            "coverage",
            "selective_accuracy",
        ]
    ]
    .copy()
    .sort_values("coverage")
)

fig, ax = plt.subplots(
    figsize=(8, 6)
)

ax.plot(
    risk["coverage"],
    1.0 - risk["selective_accuracy"],
    marker="o",
)

for _, row in risk.iterrows():
    ax.annotate(
        f"q={row['support_quantile']:.3f}",
        (
            row["coverage"],
            1.0 - row["selective_accuracy"],
        ),
        xytext=(5, 5),
        textcoords="offset points",
    )

ax.axhline(
    1.0
    - baseline_oof_metrics["accuracy"],
    linestyle="--",
    linewidth=1,
)

ax.set_xlabel(
    "Coverage (fraction classified)"
)

ax.set_ylabel(
    "Error rate among classified observations"
)

ax.set_title(
    "K5 reject option: risk–coverage tradeoff"
)

ax.grid(
    alpha=0.2
)

plt.tight_layout()

fig_path = (
    FIG_DIR
    / "DIAGNOSTIC_OOF_risk_coverage.png"
)

fig.savefig(
    fig_path,
    dpi=250,
    bbox_inches="tight",
)

plt.show()
plt.close(fig)

print(
    "Saved:",
    fig_path,
)

## Default 95% support rule on held-out observations

Status codes:

- `1` — RF-predicted class supported;
- `2` — RF-predicted class unsupported, but another K5 class supports the observation;
- `3` — unsupported by all five classes; reject candidate.

In [ ]:
# =============================================================================
# DEFAULT OOF STATUS
# =============================================================================

q = DEFAULT_SUPPORT_QUANTILE

default_diag = oof_support[q]


oof_status_df = baseline[
    [
        "PlotID",
        "Year",
        "parent_cluster_k5",
        "Easting",
        "Northing",
    ]
].copy()

oof_status_df[
    "observed_label"
] = oof_status_df[
    "parent_cluster_k5"
].map(
    K5_LABELS
)

oof_status_df[
    "rf_prediction"
] = oof_rf_pred

oof_status_df[
    "rf_prediction_label"
] = pd.Series(
    oof_rf_pred
).map(
    K5_LABELS
).to_numpy()

oof_status_df[
    "rf_correct"
] = (
    oof_rf_pred
    == y_base
)

oof_status_df[
    "support_count"
] = default_diag[
    "support_count"
]

oof_status_df[
    "min_support_ratio"
] = default_diag[
    "min_ratio"
]

oof_status_df[
    "predicted_class_supported"
] = default_diag[
    "predicted_supported"
]

oof_status_df[
    "support_status"
] = default_diag[
    "status"
]

oof_status_df[
    "support_status_label"
] = pd.Series(
    default_diag["status"]
).map(
    {
        1: "RF prediction supported",
        2: "RF unsupported; alternative class supported",
        3: "UNSUPPORTED BY ALL / reject",
    }
).to_numpy()


write_csv_safe(
    oof_status_df,
    TABLE_DIR
    / "DIAGNOSTIC_OOF_default_support_status.csv",
)


status_summary = (
    oof_status_df
    .groupby(
        [
            "support_status",
            "support_status_label",
        ],
        as_index=False,
    )
    .agg(
        n=("PlotID", "size"),
        accuracy=("rf_correct", "mean"),
        mean_min_support_ratio=(
            "min_support_ratio",
            "mean",
        ),
    )
)

status_summary[
    "fraction"
] = (
    status_summary["n"]
    / len(oof_status_df)
)

display(
    status_summary.round(4)
)

## Fit full augmented-all support geometry and map 2025

The wall-to-wall mapping reuses the existing `augmented_all_2026` hard class and uncertainty rasters. It only adds class-support diagnostics, so the RF map itself is not recomputed.

In [ ]:
# =============================================================================
# FULL SUPPORT GEOMETRY
# =============================================================================

X_full = np.vstack(
    [
        X_base,
        X_pseudo,
    ]
)

y_full = np.concatenate(
    [
        y_base,
        y_pseudo,
    ]
)


support_models_full = fit_support_models(
    X_full,
    y_full,
    support_quantile=DEFAULT_SUPPORT_QUANTILE,
)


threshold_rows = []

for code_i in K5_CODES:

    model = support_models_full[
        code_i
    ]

    threshold_rows.append(
        {
            "class_code": code_i,
            "class_label": K5_LABELS[code_i],
            "n_training": int(
                np.sum(
                    y_full
                    == code_i
                )
            ),
            "support_quantile": DEFAULT_SUPPORT_QUANTILE,
            "mahalanobis_d2_threshold": model["threshold"],
            "crossfit_d2_median": float(
                np.median(
                    model["crossfit_d2"]
                )
            ),
            "crossfit_d2_p95": float(
                np.quantile(
                    model["crossfit_d2"],
                    0.95,
                )
            ),
        }
    )


threshold_df = pd.DataFrame(
    threshold_rows
)


write_csv_safe(
    threshold_df,
    TABLE_DIR
    / "DIAGNOSTIC_full_support_thresholds.csv",
)

display(
    threshold_df.round(3)
)

In [ ]:
# =============================================================================
# WALL-TO-WALL SUPPORT MAPPING
# =============================================================================

SUPPORT_COUNT_PATH = (
    MAP_DIR
    / "2025_K5_augmented_all_support_count.tif"
)

MIN_RATIO_PATH = (
    MAP_DIR
    / "2025_K5_augmented_all_min_support_ratio.tif"
)

SUPPORT_STATUS_PATH = (
    MAP_DIR
    / "2025_K5_augmented_all_support_status.tif"
)

REJECT_CLASS_PATH = (
    MAP_DIR
    / "2025_K5_augmented_all_reject_class.tif"
)

NEAREST_SUPPORT_CLASS_PATH = (
    MAP_DIR
    / "2025_K5_augmented_all_nearest_support_class.tif"
)


for path in [
    SUPPORT_COUNT_PATH,
    MIN_RATIO_PATH,
    SUPPORT_STATUS_PATH,
    REJECT_CLASS_PATH,
    NEAREST_SUPPORT_CLASS_PATH,
]:
    if path.exists() and not ALLOW_OVERWRITE:
        raise FileExistsError(path)


src_path = annual_k2_path(
    OPTICAL_YEAR
)


with rasterio.open(
    src_path
) as src, rasterio.open(
    EXISTING_CLASS_MAP
) as class_src:

    if not grids_match(
        src,
        class_src,
    ):
        raise ValueError(
            "2025 harmonic raster and augmented-all class map are not aligned."
        )

    indexes = ia_band_indexes(
        src,
        SAMPLE_IA_COLS,
    )

    uint_profile = src.profile.copy()
    uint_profile.update(
        driver="GTiff",
        count=1,
        dtype="uint8",
        nodata=255,
        compress=COMPRESS,
        tiled=True,
        BIGTIFF="IF_SAFER",
    )

    class_profile = src.profile.copy()
    class_profile.update(
        driver="GTiff",
        count=1,
        dtype="uint8",
        nodata=0,
        compress=COMPRESS,
        tiled=True,
        BIGTIFF="IF_SAFER",
    )

    float_profile = src.profile.copy()
    float_profile.update(
        driver="GTiff",
        count=1,
        dtype="float32",
        nodata=-9999.0,
        compress=COMPRESS,
        predictor=3,
        tiled=True,
        BIGTIFF="IF_SAFER",
    )

    blocks = list(
        src.block_windows(1)
    )

    n_blocks = len(blocks)
    start = time.perf_counter()

    with rasterio.open(
        SUPPORT_COUNT_PATH,
        "w",
        **uint_profile,
    ) as count_dst, rasterio.open(
        MIN_RATIO_PATH,
        "w",
        **float_profile,
    ) as ratio_dst, rasterio.open(
        SUPPORT_STATUS_PATH,
        "w",
        **uint_profile,
    ) as status_dst, rasterio.open(
        REJECT_CLASS_PATH,
        "w",
        **class_profile,
    ) as reject_dst, rasterio.open(
        NEAREST_SUPPORT_CLASS_PATH,
        "w",
        **uint_profile,
    ) as nearest_dst:

        for block_number, (
            _,
            window,
        ) in enumerate(
            blocks,
            start=1,
        ):

            raw_features = src.read(
                indexes,
                window=window,
                masked=False,
            ).astype(
                np.float32,
                copy=False,
            )

            raw_features = np.moveaxis(
                raw_features,
                0,
                -1,
            )

            rf_class = class_src.read(
                1,
                window=window,
                masked=False,
            )

            valid = (
                rf_class
                > 0
            ) & np.isfinite(
                raw_features
            ).all(
                axis=2
            )

            if src.nodata is not None:
                valid &= ~np.any(
                    np.isclose(
                        raw_features,
                        src.nodata,
                    ),
                    axis=2,
                )

            shape = (
                int(window.height),
                int(window.width),
            )

            count_block = np.full(
                shape,
                255,
                dtype=np.uint8,
            )

            ratio_block = np.full(
                shape,
                -9999.0,
                dtype=np.float32,
            )

            status_block = np.full(
                shape,
                255,
                dtype=np.uint8,
            )

            reject_block = np.zeros(
                shape,
                dtype=np.uint8,
            )

            nearest_block = np.full(
                shape,
                255,
                dtype=np.uint8,
            )

            if valid.any():

                X_raw = raw_features[
                    valid
                ]

                X_z = (
                    X_raw
                    - MU_2025.astype(
                        np.float32
                    )
                ) / SD_2025.astype(
                    np.float32
                )

                rf_pred = rf_class[
                    valid
                ].astype(
                    np.uint8
                )

                diagnostic = support_diagnostics(
                    X_z,
                    rf_pred,
                    support_models_full,
                )

                count_block[
                    valid
                ] = diagnostic[
                    "support_count"
                ]

                ratio_block[
                    valid
                ] = diagnostic[
                    "min_support_ratio"
                ].astype(
                    np.float32
                )

                status_block[
                    valid
                ] = diagnostic[
                    "status"
                ]

                nearest_block[
                    valid
                ] = diagnostic[
                    "nearest_support_class"
                ]

                final_class = rf_pred.copy()

                final_class[
                    diagnostic[
                        "support_count"
                    ]
                    == 0
                ] = UNCLASSIFIED_CODE

                reject_block[
                    valid
                ] = final_class

            count_dst.write(
                count_block,
                1,
                window=window,
            )

            ratio_dst.write(
                ratio_block,
                1,
                window=window,
            )

            status_dst.write(
                status_block,
                1,
                window=window,
            )

            reject_dst.write(
                reject_block,
                1,
                window=window,
            )

            nearest_dst.write(
                nearest_block,
                1,
                window=window,
            )

            if (
                block_number == 1
                or block_number % 25 == 0
                or block_number == n_blocks
            ):
                elapsed = (
                    time.perf_counter()
                    - start
                )

                print(
                    f"{block_number:,}/{n_blocks:,} "
                    f"({100 * block_number / n_blocks:5.1f}%) | "
                    f"{elapsed / 60:6.1f} min"
                )

print("Wrote open-set support maps.")

## Compare open-set support status with RF uncertainty

This distinguishes two mechanisms:

- **high RF uncertainty but inside known class support**: ambiguous among known states;
- **outside all class support**: open-set / unsupported.

Particularly interesting are RF-confident pixels that are nevertheless outside all empirical supports.

In [ ]:
# =============================================================================
# SUPPORT STATUS × RF UNCERTAINTY
# =============================================================================

parts = []


with rasterio.open(
    SUPPORT_STATUS_PATH
) as status_src, rasterio.open(
    EXISTING_UNCERTAINTY_MAP
) as unc_src:

    if not grids_match(
        status_src,
        unc_src,
    ):
        raise ValueError(
            "Support-status and uncertainty rasters are not aligned."
        )

    for _, window in status_src.block_windows(1):

        status = status_src.read(
            1,
            window=window,
            masked=False,
        )

        unc = unc_src.read(
            window=window,
            masked=False,
        )

        valid = status != 255

        if not valid.any():
            continue

        pmax = unc[0][valid]
        margin = unc[1][valid]
        entropy = unc[2][valid]

        df = pd.DataFrame(
            {
                "status": status[valid],
                "pmax": pmax,
                "margin": margin,
                "normalized_entropy": (
                    entropy
                    / np.log(
                        len(K5_CODES)
                    )
                ),
            }
        )

        parts.append(
            df.groupby(
                "status",
                as_index=False,
            )
            .agg(
                n=("pmax", "size"),
                sum_pmax=("pmax", "sum"),
                sum_margin=("margin", "sum"),
                sum_entropy=(
                    "normalized_entropy",
                    "sum",
                ),
            )
        )


status_accum = pd.concat(
    parts,
    ignore_index=True,
)

status_summary = (
    status_accum
    .groupby(
        "status",
        as_index=False,
    )
    .agg(
        n=("n", "sum"),
        sum_pmax=("sum_pmax", "sum"),
        sum_margin=("sum_margin", "sum"),
        sum_entropy=("sum_entropy", "sum"),
    )
)

status_summary[
    "mean_pmax"
] = (
    status_summary["sum_pmax"]
    / status_summary["n"]
)

status_summary[
    "mean_margin"
] = (
    status_summary["sum_margin"]
    / status_summary["n"]
)

status_summary[
    "mean_normalized_entropy"
] = (
    status_summary["sum_entropy"]
    / status_summary["n"]
)

status_summary[
    "fraction_map"
] = (
    status_summary["n"]
    / status_summary["n"].sum()
)

status_summary[
    "status_label"
] = status_summary[
    "status"
].map(
    {
        1: "RF prediction supported",
        2: "RF unsupported; alternative class supported",
        3: "UNSUPPORTED BY ALL / reject",
    }
)

status_summary = status_summary[
    [
        "status",
        "status_label",
        "n",
        "fraction_map",
        "mean_pmax",
        "mean_margin",
        "mean_normalized_entropy",
    ]
]

write_csv_safe(
    status_summary,
    TABLE_DIR
    / "DIAGNOSTIC_2025_support_status_uncertainty_summary.csv",
)

display(
    status_summary.round(4)
)

## Plot the open-set maps

In [ ]:
# =============================================================================
# PLOT HELPERS + MAPS
# =============================================================================

def clipped_raster(
    raster_path,
    boundary_gdf,
    band=1,
):
    with rasterio.open(
        raster_path
    ) as src:

        boundary = boundary_gdf.to_crs(
            src.crs
        )

        arr, transform = rasterio.mask.mask(
            src,
            boundary.geometry,
            crop=True,
            indexes=band,
            filled=True,
            nodata=src.nodata,
        )

        if arr.ndim == 3:
            arr = arr[0]

        height, width = arr.shape

        west, south, east, north = array_bounds(
            height,
            width,
            transform,
        )

        extent = [
            west,
            east,
            south,
            north,
        ]

    return arr, extent


nca = gpd.read_file(
    TRUE_NCA_BOUNDARY
)


# -----------------------------------------------------------------------------
# 1. REJECT-CLASS MAP
# -----------------------------------------------------------------------------

if DISPLAY_MAPS:

    arr, extent = clipped_raster(
        REJECT_CLASS_PATH,
        nca,
        band=1,
    )

    tab10 = plt.get_cmap(
        "tab10"
    )

    cmap = ListedColormap(
        [
            tab10(2),
            tab10(0),
            tab10(1),
            tab10(4),
            tab10(3),
            "lightgray",
        ]
    )

    display_arr = np.ma.masked_where(
        arr == 0,
        arr,
    )

    fig, ax = plt.subplots(
        figsize=(12, 8)
    )

    ax.imshow(
        display_arr,
        cmap=cmap,
        vmin=1,
        vmax=6,
        extent=extent,
        origin="upper",
        interpolation="nearest",
    )

    handles = [
        Patch(
            facecolor=cmap(code_i - 1),
            label=f"{code_i} — {K5_LABELS[code_i]}",
        )
        for code_i in K5_CODES
    ]

    handles.append(
        Patch(
            facecolor="lightgray",
            label="6 — UNCLASSIFIED / unsupported",
        )
    )

    ax.legend(
        handles=handles,
        title="K5 open-set class",
        loc="center left",
        bbox_to_anchor=(1.01, 0.5),
        frameon=False,
    )

    ax.set_title(
        "2025 — K5 augmented-all classification with reject option\n"
        f"Ledoit-Wolf support · q={DEFAULT_SUPPORT_QUANTILE}",
        fontsize=15,
    )

    ax.set_axis_off()

    plt.tight_layout()

    fig_path = (
        FIG_DIR
        / "2025_K5_augmented_all_reject_class.png"
    )

    fig.savefig(
        fig_path,
        dpi=250,
        bbox_inches="tight",
    )

    plt.show()
    plt.close(fig)


# -----------------------------------------------------------------------------
# 2. SUPPORT COUNT
# -----------------------------------------------------------------------------

if DISPLAY_MAPS:

    arr, extent = clipped_raster(
        SUPPORT_COUNT_PATH,
        nca,
        band=1,
    )

    arr = np.ma.masked_where(
        arr == 255,
        arr,
    )

    fig, ax = plt.subplots(
        figsize=(12, 8)
    )

    im = ax.imshow(
        arr,
        cmap="viridis",
        vmin=0,
        vmax=5,
        extent=extent,
        origin="upper",
        interpolation="nearest",
    )

    cbar = fig.colorbar(
        im,
        ax=ax,
        fraction=0.035,
        pad=0.025,
        ticks=[0, 1, 2, 3, 4, 5],
    )

    cbar.set_label(
        "Number of K5 class supports containing pixel"
    )

    ax.set_title(
        "2025 — number of empirically supported K5 states",
        fontsize=15,
    )

    ax.set_axis_off()

    plt.tight_layout()

    fig_path = (
        FIG_DIR
        / "2025_K5_augmented_all_support_count.png"
    )

    fig.savefig(
        fig_path,
        dpi=250,
        bbox_inches="tight",
    )

    plt.show()
    plt.close(fig)


# -----------------------------------------------------------------------------
# 3. MINIMUM SUPPORT RATIO
# -----------------------------------------------------------------------------

if DISPLAY_MAPS:

    ratio_arr, extent = clipped_raster(
        MIN_RATIO_PATH,
        nca,
        band=1,
    )

    valid = (
        np.isfinite(ratio_arr)
        & (ratio_arr > 0)
        & (ratio_arr != -9999.0)
    )

    log_ratio = np.full(
        ratio_arr.shape,
        np.nan,
        dtype=np.float32,
    )

    log_ratio[
        valid
    ] = np.log10(
        ratio_arr[valid]
    )

    d = log_ratio[
        np.isfinite(log_ratio)
    ]

    vmax = float(
        np.quantile(
            np.abs(d),
            0.98,
        )
    )

    if (
        not np.isfinite(vmax)
        or vmax == 0
    ):
        vmax = 1.0

    fig, ax = plt.subplots(
        figsize=(12, 8)
    )

    im = ax.imshow(
        np.ma.masked_invalid(
            log_ratio
        ),
        cmap="RdBu_r",
        vmin=-vmax,
        vmax=vmax,
        extent=extent,
        origin="upper",
        interpolation="nearest",
    )

    cbar = fig.colorbar(
        im,
        ax=ax,
        fraction=0.035,
        pad=0.025,
    )

    cbar.set_label(
        r"$\log_{10}(R_{\min})$"
        "\n< 0 = inside at least one class support"
        "\n> 0 = outside all class supports"
    )

    ax.set_title(
        "2025 — distance from nearest empirical K5 support",
        fontsize=15,
    )

    ax.set_axis_off()

    plt.tight_layout()

    fig_path = (
        FIG_DIR
        / "2025_K5_augmented_all_min_support_ratio.png"
    )

    fig.savefig(
        fig_path,
        dpi=250,
        bbox_inches="tight",
    )

    plt.show()
    plt.close(fig)


# -----------------------------------------------------------------------------
# 4. SUPPORT STATUS
# -----------------------------------------------------------------------------

if DISPLAY_MAPS:

    arr, extent = clipped_raster(
        SUPPORT_STATUS_PATH,
        nca,
        band=1,
    )

    display_arr = np.ma.masked_where(
        arr == 255,
        arr,
    )

    status_cmap = ListedColormap(
        [
            "lightgray",
            "gold",
            "black",
        ]
    )

    fig, ax = plt.subplots(
        figsize=(12, 8)
    )

    ax.imshow(
        display_arr,
        cmap=status_cmap,
        vmin=1,
        vmax=3,
        extent=extent,
        origin="upper",
        interpolation="nearest",
    )

    handles = [
        Patch(
            facecolor="lightgray",
            label="1 — RF prediction supported",
        ),
        Patch(
            facecolor="gold",
            label="2 — RF unsupported; another class supported",
        ),
        Patch(
            facecolor="black",
            label="3 — unsupported by all / reject",
        ),
    ]

    ax.legend(
        handles=handles,
        title="Support diagnostic",
        loc="center left",
        bbox_to_anchor=(1.01, 0.5),
        frameon=False,
    )

    ax.set_title(
        "2025 — where the K5 classifier is struggling\n"
        f"Open-set support diagnostic · q={DEFAULT_SUPPORT_QUANTILE}",
        fontsize=15,
    )

    ax.set_axis_off()

    plt.tight_layout()

    fig_path = (
        FIG_DIR
        / "2025_K5_augmented_all_support_status.png"
    )

    fig.savefig(
        fig_path,
        dpi=250,
        bbox_inches="tight",
    )

    plt.show()
    plt.close(fig)

## Interpretation

The reject option should be retained only if it does something useful under held-out validation.

Evidence in favor would be:

- rejected observations have a higher error rate than retained observations;
- selective accuracy and/or macro F1 increase at reasonable coverage;
- the reject class is not overwhelmingly concentrated in one observed K5 class;
- wall-to-wall rejected pixels are spatially/ecologically coherent rather than obvious processing artifacts.

`UNCLASSIFIED` is **not a sixth ecological state**. It means only:

> the annual-z IA45 feature vector falls outside the empirical optical support of all five currently observed K5 vegetation states.

The support-count map is equally important. A pixel supported by several states is an **ambiguous known configuration**; a pixel supported by zero states is an **open-set / unsupported configuration**.

In [ ]:
# =============================================================================
# OUTPUT MANIFEST
# =============================================================================

print(
    "=" * 100
)

print(
    "K5 OPEN-SET / REJECT-OPTION DIAGNOSTIC COMPLETE"
)

print(
    "=" * 100
)

print(
    "\nOutput directory:"
)

print(
    OUTPUT_DIR
)

for label, directory in [
    ("Tables", TABLE_DIR),
    ("Maps", MAP_DIR),
    ("Figures", FIG_DIR),
]:
    print(f"\n{label}:")
    for path in sorted(directory.glob("*")):
        print(" ", path.name)